In [2]:
# Load libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow.keras import layers

# -------------------------- LOAD AND PREPROCESS DATA --------------------------

# Load the dataset 
x_data = np.load('C:/Users/hp-5c/Desktop/deep_learning/archive/x.npy')
y_data = np.load('C:/Users/hp-5c/Desktop/deep_learning/archive/y.npy')

# Normalize image data
x_data = x_data.astype("float32") / 255.0

# Print shapes
print("x_data shape:", x_data.shape)
print("y_data shape:", y_data.shape)

# Label encoding
y_labels = y_data.flatten()
classes = np.unique(y_labels)
label_dict = {label: i for i, label in enumerate(classes)}
y_int = np.array([label_dict[label] for label in y_labels])
y_cat = to_categorical(y_int, num_classes=len(label_dict)).astype(int)

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x_data, y_cat, test_size=0.2, random_state=42)

# ------------------------- SIMCLR PRETRAINING -------------------------

# Strong augmentations for SimCLR
simclr_augment = keras.Sequential([
    layers.Resizing(160, 160),  # Resize up before cropping
    layers.RandomCrop(128, 128),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

def create_simclr_pairs(x):
    return simclr_augment(x), simclr_augment(x)

def simclr_dataset(x_data, batch_size=128):
    dataset = tf.data.Dataset.from_tensor_slices(x_data)
    dataset = dataset.shuffle(1000).map(lambda x: create_simclr_pairs(x), num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

pretrain_ds = simclr_dataset(x_data)

# Encoder network
def get_encoder():
    inputs = keras.Input(shape=(128, 128, 3))
    x = layers.Conv2D(64, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, x, name="encoder")

# SimCLR model = encoder + projection head
def get_simclr_model(encoder):
    inputs = keras.Input(shape=(128, 128, 3))
    features = encoder(inputs)
    x = layers.Dense(256, activation='relu')(features)
    outputs = layers.Dense(128)(x)
    return keras.Model(inputs, outputs, name="simclr_model")

encoder = get_encoder()
simclr_model = get_simclr_model(encoder)

# Contrastive loss (NT-Xent)
def nt_xent_loss(z_i, z_j, temperature=0.5):
    z_i = tf.math.l2_normalize(z_i, axis=1)
    z_j = tf.math.l2_normalize(z_j, axis=1)
    representations = tf.concat([z_i, z_j], axis=0)
    similarity_matrix = tf.matmul(representations, representations, transpose_b=True)

    logits_mask = tf.linalg.diag(tf.zeros(tf.shape(similarity_matrix)[0])) + 1e-9
    logits = similarity_matrix / temperature
    logits = logits * (1.0 - logits_mask)

    batch_size = tf.shape(z_i)[0]
    labels = tf.range(batch_size)
    labels = tf.concat([labels, labels], axis=0)

    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
    return tf.reduce_mean(loss)

# Pretraining loop
optimizer = keras.optimizers.Adam()
@tf.function
def train_step(x1, x2):
    with tf.GradientTape() as tape:
        z_i = simclr_model(x1, training=True)
        z_j = simclr_model(x2, training=True)
        loss = nt_xent_loss(z_i, z_j)
    gradients = tape.gradient(loss, simclr_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, simclr_model.trainable_variables))
    return loss

print("Starting SimCLR Pretraining...")
for epoch in range(5):
    total_loss = 0
    batches = 0
    for x1, x2 in pretrain_ds:
        loss = train_step(x1, x2)
        total_loss += loss
        batches += 1
    print(f"Epoch {epoch+1}, Loss: {total_loss / batches:.4f}")

# ------------------------- FINE-TUNING CLASSIFIER -------------------------

print("\nFine-tuning classifier on labeled data...")

classifier = keras.Sequential([
    encoder,
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(label_dict), activation='softmax')
])

classifier.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

early_stopping = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5)

history = classifier.fit(x_train, y_train, epochs=10,
                         validation_data=(x_test, y_test),
                         callbacks=[early_stopping, reduce_lr])

# ------------------------- EVALUATION -------------------------

predictions_nn_probs = classifier.predict(x_test)
predictions_nn = np.argmax(predictions_nn_probs, axis=1)
test_labels_multiclass = np.argmax(y_test, axis=1)

accuracy = accuracy_score(test_labels_multiclass, predictions_nn)
print("Test Accuracy:", accuracy)

# Plot training curves
fig, axis = plt.subplots(1, 2, figsize=(20, 6))
axis[0].plot(history.epoch, history.history['loss'])
axis[0].plot(history.epoch, history.history['val_loss'])
axis[0].set_title("Loss")
axis[0].legend(['Train', 'Val'])

axis[1].plot(history.epoch, history.history['accuracy'])
axis[1].plot(history.epoch, history.history['val_accuracy'])
axis[1].set_title("Accuracy")
axis[1].legend(['Train', 'Val'])

plt.show()


x_data shape: (22801, 128, 128, 3)
y_data shape: (22801, 1)
Starting SimCLR Pretraining...
Epoch 1, Loss: 5.5229
Epoch 2, Loss: 5.2141
Epoch 3, Loss: 4.4632


KeyboardInterrupt: 

In [3]:
# Freeze encoder (no further updates during fine-tuning)
encoder.trainable = False

# Build full model for classification
finetune_model = keras.Sequential([
    encoder,                              # pretrained encoder from SimCLR
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(len(label_dict), activation='softmax')  # output layer
])


In [5]:
# Redefine callbacks if you're in a new cell or scope
early_stopping = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5)


In [7]:
finetune_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# You can reuse early_stopping and reduce_lr from your original code
history_finetune = finetune_model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    callbacks=[early_stopping, reduce_lr]
)


Epoch 1/10
570/570 ━━━━━━━━━━━━━━━━━━━━ 128s 218ms/step - accuracy: 0.0472 - loss: 3.2832 - val_accuracy: 0.0524 - val_loss: 3.2734 - learning_rate: 0.0010
Epoch 2/10
570/570 ━━━━━━━━━━━━━━━━━━━━ 116s 203ms/step - accuracy: 0.0544 - loss: 3.2630 - val_accuracy: 0.0517 - val_loss: 3.2560 - learning_rate: 0.0010
Epoch 3/10
570/570 ━━━━━━━━━━━━━━━━━━━━ 142s 202ms/step - accuracy: 0.0609 - loss: 3.2518 - val_accuracy: 0.0515 - val_loss: 3.2552 - learning_rate: 0.0010
Epoch 4/10
570/570 ━━━━━━━━━━━━━━━━━━━━ 115s 201ms/step - accuracy: 0.0619 - loss: 3.2468 - val_accuracy: 0.0568 - val_loss: 3.2505 - learning_rate: 0.0010
Epoch 5/10
394/570 ━━━━━━━━━━━━━━━━━━━━ 28s 162ms/step - accuracy: 0.0652 - loss: 3.2433

KeyboardInterrupt: 